# Module Education - Preparation et harmonisation

Notebook de nettoyage et preparation du domaine Education (Acces numerique, Ecoles, Resultats).
Les controles (apercu, describe, isnull) sont affiches dans les cellules.

## Sections

1. Chargement des fichiers et inventaire
2. Fonctions utilitaires de nettoyage
3. Nettoyage sous-domaine Acces numerique
4. Nettoyage sous-domaine Ecoles
5. Nettoyage sous-domaine Resultats
6. Nettoyage indicateurs nationaux (education_afristat)
7. Fusion des datasets necessaires (niveau annuel)
8. Harmonisation finale et suppression de colonnes non importantes
9. Exports vers data/Education et recapitulatif

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)

In [ ]:
ROOT = Path.cwd()
RAW_DIR = ROOT / "data_raw" / "education"
OUT_ROOT = ROOT / "data" / "Education"
OUT_ACCES = OUT_ROOT / "Acces_numerique"
OUT_ECOLES = OUT_ROOT / "Ecoles"
OUT_RESULTATS = OUT_ROOT / "Resultats"
for d in [OUT_ROOT, OUT_ACCES, OUT_ECOLES, OUT_RESULTATS]:
    d.mkdir(parents=True, exist_ok=True)

files = sorted([p for p in RAW_DIR.rglob("*.csv") if not p.name.startswith(".~lock")])
inventory = pd.DataFrame({
    "fichier": [str(p.relative_to(RAW_DIR)) for p in files],
    "taille_ko": [round(p.stat().st_size / 1024, 2) for p in files]
})
print("Apercu inventaire education:")
display(inventory)

In [ ]:
action_logs = []

def log_action(section: str, action: str, details: str) -> None:
    action_logs.append({"section": section, "action": action, "details": details})

def to_snake(text: str) -> str:
    text = str(text).strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [to_snake(c) for c in out.columns]
    return out

def normalize_text(v):
    if pd.isna(v):
        return v
    return re.sub(r"\s+", " ", str(v).strip())

def normalize_key(v):
    if pd.isna(v):
        return ""
    txt = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode("ascii")
    txt = re.sub(r"[^a-z0-9]+", " ", txt.lower()).strip()
    return re.sub(r"\s+", " ", txt)

def clean_geo_name(v):
    if pd.isna(v):
        return v
    t = normalize_text(v)
    t = re.sub(r"\b(region|province)\b", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip(" -_,")
    return t.title() if t else pd.NA

def drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    before = len(df)
    cleaned = df.dropna(how="all").copy()
    return cleaned, before - len(cleaned)

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "colonne": df.columns,
        "nb_manquants": [int(df[c].isna().sum()) for c in df.columns],
        "pct_manquants": [round(float(df[c].isna().mean()) * 100, 2) for c in df.columns]
    })
    return rep.sort_values("pct_manquants", ascending=False)

# Dictionnaire de convergence EN -> FR (et variantes FR) pour eviter les doublons d'indicateurs
INDICATEUR_MAP = {
    "mobile cellular subscriptions": "Abonnements de telephonie mobile",
    "mobile cellular telephone subscriptions": "Abonnements de telephonie mobile",
    "fixed telephone subscriptions": "Abonnements de telephonie fixe",
    "fixed telephone lines": "Abonnements de telephonie fixe",
    "individuals using the internet": "Utilisateurs d internet",
    "percentage of individuals using the internet": "Utilisateurs d internet",
    "internet users": "Utilisateurs d internet",
    "households with internet access": "Menages avec acces a internet",
    "households with a computer": "Menages disposant d un ordinateur",
    "active mobile broadband subscriptions": "Abonnements actifs internet mobile",
    "fixed broadband subscriptions": "Abonnements internet fixe",
    "international internet bandwidth": "Bande passante internet internationale"
}

def harmonize_indicator_fr(v):
    if pd.isna(v):
        return v
    src = normalize_text(v)
    key = normalize_key(src)

    if key in INDICATEUR_MAP:
        return INDICATEUR_MAP[key]

    # Regles heuristiques pour capter des variantes non exactes
    if "mobile" in key and "subscription" in key:
        return "Abonnements de telephonie mobile"
    if "fixed" in key and "telephone" in key:
        return "Abonnements de telephonie fixe"
    if "internet" in key and ("individual" in key or "user" in key):
        return "Utilisateurs d internet"
    if "broadband" in key and "mobile" in key:
        return "Abonnements actifs internet mobile"
    if "broadband" in key and "fixed" in key:
        return "Abonnements internet fixe"

    # Sinon on garde le libelle existant (FR ou autre)
    return src

KEEP_COLUMNS = [
    "domaine", "sous_domaine", "source_file",
    "indicateur", "indicateur_source",
    "categorie_1", "categorie_2", "unite", "annee", "valeur",
    "pays", "region", "commune", "sexe",
    "niveau_enseignement", "type_etablissement"
]

def build_standard_table(df: pd.DataFrame, sous_domaine: str, source_file: str) -> pd.DataFrame:
    data = normalize_columns(df)
    data, removed = drop_fully_empty_rows(data)
    log_action(sous_domaine, "supprimer_lignes_vides", f"{removed} ligne(s) retiree(s)")

    # Harmonisation des schemas classiques et ITU
    rename_map = {
        "unit": "unite",
        "date": "annee",
        "value": "valeur",
        "indicateurs": "indicateur",
        "seriesname": "indicateur",
        "seriesunits": "unite",
        "datayear": "annee",
        "datavalue": "valeur",
        "entityname": "pays",
        "rubriques": "categorie_1",
        "rubrique": "categorie_1",
        "niveaux_d_enseignement": "niveau_enseignement",
        "type_d_etablissement": "type_etablissement"
    }
    data = data.rename(columns=rename_map)

    # Colonnes parasites ITU de type dataValue 1, dataValue 2...
    drop_like = [c for c in data.columns if re.match(r"^datavalue_\d+$", c)]
    if drop_like:
        data = data.drop(columns=drop_like)

    if "indicateur" not in data.columns:
        data["indicateur"] = pd.NA
    if "categorie_1" not in data.columns:
        data["categorie_1"] = data.get("seriesparent", pd.NA)
    if "categorie_2" not in data.columns:
        data["categorie_2"] = pd.NA
    if "commune" not in data.columns:
        data["commune"] = pd.NA
    if "region" not in data.columns:
        data["region"] = pd.NA
    if "pays" not in data.columns:
        data["pays"] = pd.NA
    if "sexe" not in data.columns:
        data["sexe"] = pd.NA
    if "niveau_enseignement" not in data.columns:
        data["niveau_enseignement"] = pd.NA
    if "type_etablissement" not in data.columns:
        data["type_etablissement"] = pd.NA

    for col in ["indicateur", "categorie_1", "categorie_2", "unite", "pays", "commune", "sexe", "niveau_enseignement", "type_etablissement"]:
        data[col] = data[col].apply(normalize_text)
    data["region"] = data["region"].apply(clean_geo_name)

    data["annee"] = pd.to_numeric(data.get("annee"), errors="coerce")
    data["valeur"] = pd.to_numeric(data.get("valeur"), errors="coerce").fillna(0)
    data["pays"] = data["pays"].fillna("Burkina Faso")

    # Harmonisation des valeurs de cellule pour les indicateurs
    data["indicateur_source"] = data["indicateur"]
    data["indicateur"] = data["indicateur"].apply(harmonize_indicator_fr)
    log_action(sous_domaine, "harmoniser_indicateurs", "convergence des libelles EN/FR vers un libelle canonique")

    out = pd.DataFrame({
        "domaine": "Education",
        "sous_domaine": sous_domaine,
        "source_file": source_file,
        "indicateur": data["indicateur"],
        "indicateur_source": data["indicateur_source"],
        "categorie_1": data["categorie_1"],
        "categorie_2": data["categorie_2"],
        "unite": data["unite"],
        "annee": data["annee"],
        "valeur": data["valeur"],
        "pays": data["pays"],
        "region": data["region"],
        "commune": data["commune"],
        "sexe": data["sexe"],
        "niveau_enseignement": data["niveau_enseignement"],
        "type_etablissement": data["type_etablissement"]
    })

    log_action(sous_domaine, "uniformiser_colonnes_et_types", "schema standard + types harmonises")
    return out[KEEP_COLUMNS].copy()# Normalisation francaise appliquee au moment des exportsCOLONNES_FR_MAP = {    "source_file": "fichier_source",    "source_dataset": "jeu_source",    "event_id": "id_evenement",    "country": "pays",    "year": "annee",    "location": "localisation",    "event_type": "type_evenement",    "sub_event_type": "sous_type_evenement",    "event_count": "nombre_evenements",    "fatalities_total": "deces_totaux",    "fatalities_civilians": "deces_civils",    "event_date": "date_evenement",    "granularity": "granularite",}VALEURS_FR_MAP = {    "week": "hebdomadaire",    "event": "evenement",    "acled_weekly_aggregated": "acled_agrege_hebdomadaire",    "ucdp_like_conflict_events": "evenements_conflits_type_ucdp",}def normaliser_dataframe_fr(df: pd.DataFrame) -> pd.DataFrame:    out = df.copy()    out = out.rename(columns={c: COLONNES_FR_MAP.get(c, c) for c in out.columns})    for c in out.columns:        if out[c].dtype == "object":            out[c] = out[c].map(lambda v: VALEURS_FR_MAP.get(str(v).strip().lower(), v) if pd.notna(v) else v)    return out# Renforcement: traduction des valeurs de cellules (expressions + mots) vers le francaisPHRASES_REMPLACEMENTS_FR = {    "individuals using the internet": "Utilisateurs d internet",    "percentage of individuals using the internet": "Utilisateurs d internet",    "internet users": "Utilisateurs d internet",    "mobile cellular subscriptions": "Abonnements de telephonie mobile",    "mobile cellular telephone subscriptions": "Abonnements de telephonie mobile",    "fixed telephone subscriptions": "Abonnements de telephonie fixe",    "fixed telephone lines": "Abonnements de telephonie fixe",    "active mobile broadband subscriptions": "Abonnements actifs internet mobile",    "fixed broadband subscriptions": "Abonnements internet fixe",    "international internet bandwidth": "Bande passante internet internationale",    "acled_weekly_aggregated": "acled agrege hebdomadaire",    "ucdp_like_conflict_events": "evenements conflits type ucdp",}MOTS_REMPLACEMENTS_FR = {    "weekly": "hebdomadaire",    "annual": "annuel",    "event": "evenement",    "events": "evenements",    "conflict": "conflit",    "fatalities": "deces",    "fatality": "deces",    "civilian": "civil",    "civilians": "civils",    "reported": "rapporte",    "targeting": "ciblant",    "number": "nombre",    "country": "pays",    "rate": "taux",    "incidence": "incidence",    "mobile": "mobile",    "fixed": "fixe",    "broadband": "internet",    "subscriptions": "abonnements",    "subscription": "abonnement",    "data": "donnees",    "public": "public",    "private": "prive",}def _traduire_texte_fr(v):    if pd.isna(v):        return v    txt = str(v)    low = txt.lower().strip()    if low in PHRASES_REMPLACEMENTS_FR:        return PHRASES_REMPLACEMENTS_FR[low]    out = txt    for en, fr in MOTS_REMPLACEMENTS_FR.items():        out = re.sub(rf"\b{re.escape(en)}\b", fr, out, flags=re.IGNORECASE)    out = re.sub(r"\s+", " ", out).strip()    return outdef normaliser_dataframe_fr(df: pd.DataFrame) -> pd.DataFrame:    out = df.copy()    out = out.rename(columns={c: COLONNES_FR_MAP.get(c, c) for c in out.columns})    for c in out.columns:        if out[c].dtype == "object":            out[c] = out[c].map(_traduire_texte_fr)    return out

In [ ]:
# 1) Sous-domaine Acces numerique
acces_dir = RAW_DIR / "acces_numerique"
acces_frames = []
for p in sorted(acces_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not acces_frames:
        print("Apercu brut Acces numerique:")
        display(raw.head(3))
    acces_frames.append(build_standard_table(raw, "Acces_numerique", p.name))

acces_clean = pd.concat(acces_frames, ignore_index=True)
print("Apercu nettoye Acces numerique:")
display(acces_clean.head(8))
print("Correspondances indicateurs source -> indicateur harmonise (exemples):")
corresp_acces = acces_clean[["indicateur_source", "indicateur"]].drop_duplicates()
display(corresp_acces.head(15))
print("Describe Acces numerique:")
display(acces_clean[["annee", "valeur"]].describe().T)
print("isnull Acces numerique:")
display(acces_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 2) Sous-domaine Ecoles
ecoles_dir = RAW_DIR / "ecoles_fermes-ouvertes"
ecoles_frames = []
for p in sorted(ecoles_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not ecoles_frames:
        print("Apercu brut Ecoles:")
        display(raw.head(3))
    ecoles_frames.append(build_standard_table(raw, "Ecoles", p.name))

ecoles_clean = pd.concat(ecoles_frames, ignore_index=True)
print("Apercu nettoye Ecoles:")
display(ecoles_clean.head(8))
print("Describe Ecoles:")
display(ecoles_clean[["annee", "valeur"]].describe().T)
print("isnull Ecoles:")
display(ecoles_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 3) Sous-domaine Resultats
res_dir = RAW_DIR / "resultats_scolaires"
res_frames = []
for p in sorted(res_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not res_frames:
        print("Apercu brut Resultats:")
        display(raw.head(3))
    res_frames.append(build_standard_table(raw, "Resultats", p.name))

resultats_clean = pd.concat(res_frames, ignore_index=True)
print("Apercu nettoye Resultats:")
display(resultats_clean.head(8))
print("Describe Resultats:")
display(resultats_clean[["annee", "valeur"]].describe().T)
print("isnull Resultats:")
display(resultats_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 4) Indicateurs nationaux (education_afristat)
edu_afristat_raw = pd.read_csv(RAW_DIR / "education_afristat.csv")
print("Apercu brut education_afristat:")
display(edu_afristat_raw.head(3))

edu_afristat_clean = build_standard_table(edu_afristat_raw, "National", "education_afristat.csv")
print("Apercu nettoye education_afristat:")
display(edu_afristat_clean.head(8))
print("Describe education_afristat:")
display(edu_afristat_clean[["annee", "valeur"]].describe().T)
print("isnull education_afristat:")
display(edu_afristat_clean.isnull().sum().to_frame("nb_manquants"))

## Fusion des datasets necessaires

Fusion annuelle des sous-domaines sur la cle annee pour un tableau macro comparatif.

In [ ]:
annual_acces = acces_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_acces"})
annual_ecoles = ecoles_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_ecoles"})
annual_resultats = resultats_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_resultats"})
annual_nat = edu_afristat_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_nationale"})

fusion_annuelle = annual_acces.merge(annual_ecoles, on="annee", how="outer").merge(
    annual_resultats, on="annee", how="outer"
).merge(annual_nat, on="annee", how="outer")
for c in ["valeur_acces", "valeur_ecoles", "valeur_resultats", "valeur_nationale"]:
    fusion_annuelle[c] = pd.to_numeric(fusion_annuelle[c], errors="coerce").fillna(0)

log_action("fusion_datasets", "merge_annee", "fusion annuelle acces/ecoles/resultats/national")

print("Apercu fusion annuelle:")
display(fusion_annuelle.sort_values("annee").head(15))
print("Describe fusion annuelle:")
display(fusion_annuelle.describe().T)
print("isnull fusion annuelle:")
display(fusion_annuelle.isnull().sum().to_frame("nb_manquants"))

# Harmonisation finale et suppression des colonnes non importantes
education_harmonisee_full = pd.concat([acces_clean, ecoles_clean, resultats_clean, edu_afristat_clean], ignore_index=True)
dropped_columns = sorted(set(education_harmonisee_full.columns) - set(KEEP_COLUMNS))
education_harmonisee = education_harmonisee_full[KEEP_COLUMNS].copy()
education_harmonisee = education_harmonisee.sort_values(["sous_domaine", "annee", "indicateur"], na_position="last")

log_action("harmonisation_finale", "supprimer_colonnes_inutiles", f"colonnes retirees: {dropped_columns}")

print("Colonnes supprimees:")
display(pd.DataFrame({"colonnes_supprimees": dropped_columns}))
print("Apercu table education_harmonisee:")
display(education_harmonisee.head(12))
print("Describe education_harmonisee:")
display(education_harmonisee[["annee", "valeur"]].describe().T)
print("isnull education_harmonisee:")
display(education_harmonisee.isnull().sum().to_frame("nb_manquants"))
display((education_harmonisee.isnull().mean() * 100).round(2).to_frame("pct_manquants"))

# Exports
normaliser_dataframe_fr(acces_clean).to_csv(OUT_ACCES / "education_acces_numerique_harmonise.csv", index=False)
normaliser_dataframe_fr(ecoles_clean).to_csv(OUT_ECOLES / "education_ecoles_harmonise.csv", index=False)
normaliser_dataframe_fr(resultats_clean).to_csv(OUT_RESULTATS / "education_resultats_harmonise.csv", index=False)
normaliser_dataframe_fr(edu_afristat_clean).to_csv(OUT_ROOT / "education_national_harmonise.csv", index=False)
normaliser_dataframe_fr(fusion_annuelle).to_csv(OUT_ROOT / "education_fusion_annuelle.csv", index=False)
normaliser_dataframe_fr(education_harmonisee).to_csv(OUT_ROOT / "education_harmonisee_global.csv", index=False)

actions_df = pd.DataFrame(action_logs)
print("Journal des actions:")
display(actions_df)

summary = {
    "domaine": "Education",
    "nb_lignes_acces": int(len(acces_clean)),
    "nb_lignes_ecoles": int(len(ecoles_clean)),
    "nb_lignes_resultats": int(len(resultats_clean)),
    "nb_lignes_national": int(len(edu_afristat_clean)),
    "nb_lignes_harmonise_global": int(len(education_harmonisee)),
    "fichiers_generes": [
        "data/Education/Acces_numerique/education_acces_numerique_harmonise.csv",
        "data/Education/Ecoles/education_ecoles_harmonise.csv",
        "data/Education/Resultats/education_resultats_harmonise.csv",
        "data/Education/education_national_harmonise.csv",
        "data/Education/education_fusion_annuelle.csv",
        "data/Education/education_harmonisee_global.csv"
    ]
}

with open(OUT_ROOT / "synthese_preparation_education.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

## Recapitulatif

Ce notebook nettoie les fichiers bruts du module Education, standardise les colonnes, corrige les types et les valeurs manquantes, harmonise les sous-domaines, realise une fusion annuelle utile, supprime les colonnes non importantes, affiche les controles en cellules (apercu/describe/isnull), puis exporte les jeux harmonises dans data/Education.